In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn import neighbors

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

# TABLA FINAL

## Calculo de Y

In [2]:
data = pd.read_csv("final_data.csv")
data.reset_index()
data.shape

(97916, 26)

In [3]:
y = data["media_review_score"]
data = data.drop(columns = "media_review_score")
y.unique()

array([5, 4, 1, 3, 2], dtype=int64)

In [4]:
data.columns

Index(['Unnamed: 0', 'order_id', 'fecha_ultima_review',
       'order_purchase_timestamp', 'delivered_status', 'delay_time',
       'customer_unique_id', 'customer_state', 'number_payments',
       'payment_value_sum', 'payment_type', 'number_items', 'total_price',
       'total_freight_value', 'total_diff_items', 'product_id', 'seller_id',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english', 'seller_state'],
      dtype='object')

## ANALISIS

In [5]:
# Para un modelo de tipo KNN descartaremos las fechas, ids, y dada la baja relacion entre el estado y
# la satisfaccion se descartara los estados.
data = data.drop(columns = ["fecha_ultima_review", 
                            "order_id",
                            "order_purchase_timestamp",
                            "product_id",
                            "seller_id",
                            "seller_state",
                            "customer_state",
                            "customer_unique_id",
                            "Unnamed: 0"])
data.dtypes

delivered_status                   int64
delay_time                         int64
number_payments                  float64
payment_value_sum                float64
payment_type                      object
number_items                     float64
total_price                      float64
total_freight_value              float64
total_diff_items                 float64
product_description_lenght       float64
product_photos_qty               float64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
product_category_name_english     object
dtype: object

In [6]:
data.duplicated().sum()

2839

In [7]:
data.isna().sum()

delivered_status                 0
delay_time                       0
number_payments                  0
payment_value_sum                0
payment_type                     0
number_items                     0
total_price                      0
total_freight_value              0
total_diff_items                 0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
dtype: int64

In [8]:
data.dtypes

delivered_status                   int64
delay_time                         int64
number_payments                  float64
payment_value_sum                float64
payment_type                      object
number_items                     float64
total_price                      float64
total_freight_value              float64
total_diff_items                 float64
product_description_lenght       float64
product_photos_qty               float64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
product_category_name_english     object
dtype: object

In [9]:
cols_int = ["number_payments","number_items","total_diff_items", "product_description_lenght", "product_photos_qty"]
data[cols_int] = data[cols_int].astype("int64")
data.dtypes

delivered_status                   int64
delay_time                         int64
number_payments                    int64
payment_value_sum                float64
payment_type                      object
number_items                       int64
total_price                      float64
total_freight_value              float64
total_diff_items                   int64
product_description_lenght         int64
product_photos_qty                 int64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
product_category_name_english     object
dtype: object

In [10]:
# Hacer histplot mas legibles
# sns.countplot(data, x = "delivered_status", hue = "review_category")
# plt.show()
# sns.countplot(data, x = "payment_type", hue = "review_category")
# plt.show()
# sns.countplot(data, x = "number_items", hue = "review_category")
# plt.show()
# sns.countplot(data, x = "total_diff_items", hue = "review_category")
# plt.show()
# sns.countplot(data, x = "product_photos_qty", hue = "review_category")
# plt.show()
# sns.countplot(data, x = "product_category_name_english", hue = "review_category")
# plt.xticks(rotation=90)  # <- AHORA VERTICAL
# plt.show()

# sns.histplot(data, x = "payment_value_sum", hue = "review_category")
# plt.xlim(0, 500)
# plt.show()
# sns.histplot(data, x = "total_price", hue = "review_category")
# plt.xlim(0, 500)
# plt.show()
# sns.histplot(data, x = "total_freight_value", hue = "review_category")
# plt.xlim(0, 500)
# plt.show()
# sns.histplot(data, x = "product_description_lenght", hue = "review_category")
# plt.show()
# sns.histplot(data, x = "product_weight_g", hue = "review_category")
# plt.show()
# sns.histplot(data, x = "product_length_cm", hue = "review_category")
# plt.show()
# sns.histplot(data, x = "product_height_cm", hue = "review_category")
# plt.show()
# sns.histplot(data, x = "product_width_cm", hue = "review_category")
# plt.show()
# sns.histplot(data, x = "delay_time", hue = "review_category")
# plt.show()

In [11]:
data.columns

Index(['delivered_status', 'delay_time', 'number_payments',
       'payment_value_sum', 'payment_type', 'number_items', 'total_price',
       'total_freight_value', 'total_diff_items', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm',
       'product_category_name_english'],
      dtype='object')

## MAPEO DE VARIABLES CATEGORICAS Y ESCALADO

In [12]:
for col in data.columns:
    print(f"{col}: diff= {data[col].nunique()},  max= {data[col].max()},  min= {data[col].min()}")

delivered_status: diff= 2,  max= 1,  min= 0
delay_time: diff= 197,  max= 147,  min= -200
number_payments: diff= 20,  max= 29,  min= 1
payment_value_sum: diff= 27474,  max= 13664.08,  min= 9.59
payment_type: diff= 5,  max= voucher,  min= boleto
number_items: diff= 17,  max= 21,  min= 1
total_price: diff= 7587,  max= 13440.0,  min= 0.85
total_freight_value: diff= 7666,  max= 1794.96,  min= 0.0
total_diff_items: diff= 8,  max= 8,  min= 1
product_description_lenght: diff= 2952,  max= 3992,  min= 4
product_photos_qty: diff= 19,  max= 20,  min= 1
product_weight_g: diff= 2188,  max= 40425.0,  min= 0.0
product_length_cm: diff= 99,  max= 105.0,  min= 7.0
product_height_cm: diff= 102,  max= 105.0,  min= 2.0
product_width_cm: diff= 95,  max= 118.0,  min= 6.0
product_category_name_english: diff= 6,  max= Tools and Construction,  min= Electronics and Technology


In [13]:
# Mapeo de categorias
mapeo_categorias = {
    "Home and Decoration":0,
    "Electronics and Technology":1,
    "Fashion and Personal Care":2,
    "Leisure, Toys and Arts":3,
    "Tools and Construction":4,
    "Miscellaneous and Other Items":5
}

data["product_category_name_english"] = data["product_category_name_english"].map(mapeo_categorias)
data["product_category_name_english"].nunique()
# MinMax de todas las variables 

6

In [14]:
data["payment_type"].isna().sum()

0

In [15]:
data["payment_type"].unique()

array(['credit_card', 'boleto', 'multiple_payments', 'debit_card',
       'voucher'], dtype=object)

In [16]:
# Mapeo de pagos
payment_map = {
    "multiple_payments": 4,
    "credit_card": 3,
    "debit_card": 2,
    "boleto": 1,
    "voucher": 0
}
data["payment_type"] = data["payment_type"].map(payment_map)
data["payment_type"].nunique()

5

In [17]:
data["payment_type"].isna().sum()

0

In [18]:
data.head()

,delivered_status,delay_time,number_payments,payment_value_sum,payment_type,number_items,total_price,total_freight_value,total_diff_items,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1,9,1,72.19,3,1,58.90,13.29,1,598,4,650.0,28.0,9.0,14.0,5
1,1,3,1,259.83,3,1,239.90,19.93,1,239,2,30000.0,50.0,30.0,40.0,5
2,1,14,1,216.87,3,1,199.00,17.87,1,695,2,3050.0,33.0,13.0,33.0,0
3,1,6,1,25.78,3,1,12.99,12.79,1,480,1,200.0,16.0,10.0,15.0,2
4,1,16,1,218.04,3,1,199.90,18.14,1,409,1,3750.0,35.0,40.0,30.0,4


In [19]:
scaler = MinMaxScaler()
for col in data.columns:
    data[col] = scaler.fit_transform(data[[col]])
    print(f"{col} - val_diff: {data[col].nunique()} - val max: {data[col].max()} - val min: {data[col].min()}")

delivered_status - val_diff: 2 - val max: 1.0 - val min: 0.0
delay_time - val_diff: 197 - val max: 1.0 - val min: 0.0
number_payments - val_diff: 20 - val max: 0.9999999999999999 - val min: 0.0
payment_value_sum - val_diff: 27458 - val max: 1.0000000000000002 - val min: 0.0
payment_type - val_diff: 5 - val max: 1.0 - val min: 0.0
number_items - val_diff: 17 - val max: 1.0 - val min: 0.0
total_price - val_diff: 7585 - val max: 0.9999999999999999 - val min: 0.0
total_freight_value - val_diff: 7654 - val max: 1.0 - val min: 0.0
total_diff_items - val_diff: 8 - val max: 1.0 - val min: 0.0
product_description_lenght - val_diff: 2952 - val max: 0.9999999999999999 - val min: 0.0
product_photos_qty - val_diff: 19 - val max: 1.0 - val min: 0.0
product_weight_g - val_diff: 2188 - val max: 1.0 - val min: 0.0
product_length_cm - val_diff: 99 - val max: 1.0 - val min: 0.0
product_height_cm - val_diff: 102 - val max: 1.0 - val min: 0.0
product_width_cm - val_diff: 95 - val max: 1.0 - val min: 0.0
pr

In [20]:
data.head()

,delivered_status,delay_time,number_payments,payment_value_sum,payment_type,number_items,total_price,total_freight_value,total_diff_items,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1.0,0.602305,0.0,0.004585,0.75,0.0,0.004319,0.007404,0.0,0.148947,0.157895,0.016079,0.214286,0.067961,0.071429,1.0
1,1.0,0.585014,0.0,0.018327,0.75,0.0,0.017788,0.011103,0.0,0.058927,0.052632,0.742115,0.438776,0.271845,0.303571,1.0
2,1.0,0.616715,0.0,0.015180,0.75,0.0,0.014744,0.009956,0.0,0.173270,0.052632,0.075448,0.265306,0.106796,0.241071,0.0
3,1.0,0.593660,0.0,0.001186,0.75,0.0,0.000903,0.007126,0.0,0.119358,0.000000,0.004947,0.091837,0.077670,0.080357,0.4
4,1.0,0.622478,0.0,0.015266,0.75,0.0,0.014811,0.010106,0.0,0.101555,0.000000,0.092764,0.285714,0.368932,0.214286,0.8


In [21]:
data.dtypes

delivered_status                 float64
delay_time                       float64
number_payments                  float64
payment_value_sum                float64
payment_type                     float64
number_items                     float64
total_price                      float64
total_freight_value              float64
total_diff_items                 float64
product_description_lenght       float64
product_photos_qty               float64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
product_category_name_english    float64
dtype: object

In [22]:
data.isna().sum()

delivered_status                 0
delay_time                       0
number_payments                  0
payment_value_sum                0
payment_type                     0
number_items                     0
total_price                      0
total_freight_value              0
total_diff_items                 0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
dtype: int64

## CALCULO DE TRAIN Y TEST

In [23]:
data.columns

Index(['delivered_status', 'delay_time', 'number_payments',
       'payment_value_sum', 'payment_type', 'number_items', 'total_price',
       'total_freight_value', 'total_diff_items', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm',
       'product_category_name_english'],
      dtype='object')

In [25]:
X = data
X.shape

(97916, 16)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size = 0.75,
    test_size = 0.25
)
X_train.shape

(73437, 16)

# CREACION DE MODELO KNN

## Modelo

In [27]:
modelo = neighbors.KNeighborsClassifier(
    n_neighbors = 5
)
modelo.fit(X_train, y_train)
neighbors = modelo.kneighbors(X_test, return_distance = False)
neighbors

C:\Users\jacin\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\jacin\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\jacin\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jacin\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\jacin\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _wina

array([[44431, 27544, 27558,  8266, 10556],
       [65083, 55315, 11704,  5521, 53776],
       [ 3550, 43287, 69203, 14755,  4976],
       ...,
       [67822, 73114, 57160, 11992, 11922],
       [24574, 28246, 26158, 14186, 55877],
       [27056,   545, 55579, 29573, 13262]], dtype=int64)

In [28]:
# Realizamos la prediccion
y_pred = modelo.predict(X_test)
y_pred

array([5, 5, 1, ..., 5, 5, 3], dtype=int64)

In [29]:
# MAtriz de confusion para revisar la calidad del modelo
matriz_confusion = confusion_matrix(y_test, y_pred)

TN = matriz_confusion[0][0]
TP = matriz_confusion[1][1]
FN = matriz_confusion[1][0]
FP = matriz_confusion[0][1]

matriz_confusion

array([[  867,    38,    98,   374,  1287],
       [  153,     5,    33,   119,   484],
       [  200,    14,    81,   356,  1360],
       [  272,    34,   230,   779,  3396],
       [  610,    82,   583,  2193, 10831]], dtype=int64)

In [30]:
(TN + TP) / (TN + TP + FN + FP) # Accuracy

0.8203198494825964

In [31]:
TP / (TP + FN) # Sensibility

0.03164556962025317

In [32]:
TN / (TN + FP) # Specificity

0.958011049723757